This jupyter file is worked on the kaggle notebook

with dataset link: 

https://www.kaggle.com/datasets/kushagrapandya/visdrone-dataset

Force to installed the newest pytorch to adapt the newest module of ultralystics

In [ ]:

!pip install ultralytics --upgrade
!pip install Pillow --upgrade
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --force-reinstall

In [1]:
import os
import shutil
import random

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO,YOLOWorld
from kaggle_secrets import UserSecretsClient

# **Train phase**

In [2]:
src_dir = Path('/kaggle/input/datasets/kushagrapandya/visdrone-dataset/')
dst_dir = Path('/kaggle/working/VisDrone')

Actually in the VisDrone.yaml already has the information about how to download the dataset and how to applicate the convert the raw dataset to the yolo format dataset, in follow cell, we just repeat the operation like conversion the data set and set the path and classes to the model.

In [3]:

def visdrone2yolo(src_dir, dst_dir, num_samples=None):
    
    
    def convert_box(size, box):
        dw = 1. / size[0]
        dh = 1. / size[1]
        return (box[0] + box[2] / 2) * dw, (box[1] + box[3] / 2) * dh, box[2] * dw, box[3] * dh

    (dst_dir / 'labels').mkdir(parents=True, exist_ok=True)
    all_files = list((src_dir / 'annotations').glob('*.txt'))
    
    if num_samples:
        all_files = random.sample(all_files, min(num_samples, len(all_files)))
    
    for f in tqdm(all_files, desc=f'Converting {src_dir}'):
        img_size = Image.open((src_dir / 'images' / f.name).with_suffix('.jpg')).size
        lines = []
        with open(f, 'r') as file:
            for row in [x.split(',') for x in file.read().strip().splitlines()]:
                if row[4] == '0':
                    continue
                cls = int(row[5]) - 1
                box = convert_box(img_size, tuple(map(int, row[:4])))
                lines.append(f"{cls} {' '.join(f'{x:.6f}' for x in box)}\n")
        with open(dst_dir / 'labels' / f.name, 'w') as fl:
            fl.writelines(lines)

datasets = ['VisDrone2019-DET-train', 'VisDrone2019-DET-val', 'VisDrone2019-DET-test-dev']


for d in datasets:
    visdrone2yolo(src_dir / d / d, dst_dir / d / d)

Converting /kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-train/VisDrone2019-DET-train: 100%|██████████| 6471/6471 [01:06<00:00, 97.53it/s] 
Converting /kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-val/VisDrone2019-DET-val: 100%|██████████| 548/548 [00:05<00:00, 105.37it/s]
Converting /kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:24<00:00, 65.28it/s]


Copy from train/validation/test images from input workspace to working workspace

In [4]:

for d in datasets:
    src_imgs = src_dir / d / d / 'images'
    dst_imgs = dst_dir / d / d / 'images'
    
    if src_imgs.exists():
        if dst_imgs.is_symlink():
            dst_imgs.unlink()
        elif dst_imgs.is_dir():
            shutil.rmtree(dst_imgs)
            
    # recreate
    dst_imgs.mkdir(parents=True, exist_ok=True)
    
    # compy one by one
    imgs = list(src_imgs.glob('*.jpg'))
    for img in tqdm(imgs, desc=f'Copying: {d}'):
        shutil.copy(img, dst_imgs / img.name)

print("\n End")

1. delete symbolic link...


Copying: VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:02<00:00, 668.65it/s]


 End


Define new yaml file for train

In [5]:
yaml_content = """
path: /kaggle/working/VisDrone
test: VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images
train: VisDrone2019-DET-train/VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/VisDrone2019-DET-val/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/kaggle/working/VisDrone.yaml', 'w') as f:
    f.write(yaml_content)

Run command from the command line to enable the dual gpu usage offered by kaggle notebook

In [6]:
!yolo train data=/kaggle/working/VisDrone.yaml model=yolov8s-worldv2.pt epochs=100 imgsz=640 device=0,1 batch=32 project=runs name=visdrone_exp1 save_period=10

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/VisDrone.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-worldv2.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_ex

Export result as the zip file and download it, also can just download the best checkpoint file, for test.

In [7]:
shutil.make_archive('/kaggle/working/weight_dir', 'zip', '/kaggle/working/runs/detect/runs/visdrone_exp1')

'/kaggle/working/weight_dir.zip'

Save as a new kaggle dataset, can just download from the output dir, not neccesary to run following two cells.

In [27]:
import json
#os.makedirs('/kaggle/working/dataset_to_export', exist_ok=True)
#shutil.copytree('/kaggle/working/runs/detect/runs/visdrone_exp1', '/kaggle/working/dataset_to_export/results')
username = "your kaggle username"
metadata = {
    "title": "YW-trained-weights-dir",
    "id": f"{username}/YW-trained-weights-dir",
    "licenses": [{"name": "CC0-1.0"}]
}

with open('/kaggle/working/dataset_to_export/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)


In [28]:
!kaggle datasets create -p /kaggle/working/dataset_to_export --dir-mode tar

Starting upload for file results.tar
100%|████████████████████████████████████████| 792M/792M [00:10<00:00, 78.2MB/s]
Upload successful: results.tar (792MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/penglufly/YW-trained-weights-dir
